# 06 - 支持向量机 (SVM)

## 学习目标
- 理解 SVM 的最大间隔分类思想
- 掌握核函数的作用
- 实践 SVC 和 SVR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, classification_report

## 1. 线性 SVM

In [ ]:
# 生成二分类数据
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_informative=2, random_state=42, n_clusters_per_class=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 线性核
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
print(f'线性核准确率: {svm_linear.score(X_test_scaled, y_test):.4f}')

## 2. 不同核函数比较

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results = []

for kernel in kernels:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_train_scaled, y_train)
    acc = svm.score(X_test_scaled, y_test)
    results.append({'Kernel': kernel, 'Accuracy': acc})
    print(f'{kernel:10s}: {acc:.4f}')

## 3. 决策边界可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, kernel in enumerate(['linear', 'poly', 'rbf', 'sigmoid']):
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    # 创建网格
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = svm.predict(scaler.transform(np.c_[xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    axes[idx].scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k', s=30)
    axes[idx].set_title(f'{kernel} kernel')

plt.tight_layout()
plt.show()

## 4. 参数调优

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(SVC(), param_grid, cv=5, n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f'最佳参数: {grid.best_params_}')
print(f'最佳分数: {grid.best_score_:.4f}')

## 小结
- SVM 寻找最大间隔超平面
- 核函数将数据映射到高维空间
- C 控制软间隔，gamma 控制 RBF 核的影响范围

---
下一个: [07_model_evaluation.ipynb](07_model_evaluation.ipynb)